# Extract the Entities of the 1914 Press

Which people, places and organizations does a passage name? Three methods, from the fastest to the most flexible, compared on 40 passages annotated by an LLM, each entity checked against its text (in French, like the corpus):

1. spaCy, its fixed labels
2. GLiNER, labels of your choice
3. An LLM filling in a schema
4. Link names to a reference list
5. Compare, and spot inventions

This notebook runs faster with a GPU: *Runtime > Change runtime type > T4 GPU*. Without one, it still runs: a smaller LLM (Qwen3-1.7B instead of Qwen3-4B), slower on a CPU, then extracts 5 of the 40 passages only; what follows loads the extractions of the 40 made by Qwen3-4B on a GPU.

The limit: this reference itself comes from an LLM. It tells which entities really are in the text, not which ones are missing, and it may favor an LLM over spaCy and GLiNER. For a real project, annotate by hand.

## Data and Tools

The LLM runs in [Ollama](https://ollama.com), on the Colab machine: the next cell installs it and downloads the model in the background, while you work on spaCy and GLiNER.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/gold.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/teacher-gold.jsonl
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q gliner rapidfuzz
!python -m spacy download fr_core_news_md > /dev/null

import os
import subprocess

import torch

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# On a GPU, Qwen3-4B. On a CPU, Qwen3-1.7B, twice as fast: a "hybrid" model, which
# thinks before answering, except with reasoning_effort="none" (in the calls).
MODEL = "qwen3:4b-instruct-2507-q4_K_M" if GPU else "qwen3:1.7b"
# On a GPU, 8 requests at a time; on a CPU, one: the prompts' common start (the
# instructions) is then computed only once.
parallel = "8" if GPU else "1"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": parallel, "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(
  ["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
pull = subprocess.Popen(
  f"sleep 5 && ollama pull {MODEL}",
  shell=True,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
)

In [ ]:
import json

import pandas as pd
from rapidfuzz import fuzz

gold = pd.read_json("gold.jsonl", lines=True)
FIELDS = ["people", "places", "organizations"]
print(len(gold), "annotated passages")
gold.head(3)

`evaluate(predictions)` compares extractions (a list of dictionaries `{"people": [...], "places": [...], "organizations": [...]}`, in `gold`'s order) with the annotations: for each field, the precision (the share of extracted entities that are right), the recall (the share of expected entities that are found) and their F1. Matching is lenient: two names match if their similarity (`fuzz.ratio`, an edit distance without substitution) is at least 85 out of 100, or if one contains the other ("Jaurès" and "Jean Jaurès").

In [ ]:
def normalize(name: str) -> str:
  return " ".join(name.lower().replace("’", "'").split())


def same(a: str, b: str) -> bool:
  a, b = normalize(a), normalize(b)
  return fuzz.ratio(a, b) >= 85 or (min(len(a), len(b)) >= 4 and (a in b or b in a))


def evaluate(predictions: list[dict[str, list[str]]]) -> pd.DataFrame:
  rows = {}
  for field in FIELDS:
    found = expected = right_found = right_expected = 0
    for prediction, (_, row) in zip(predictions, gold.iterrows()):
      predicted = list(dict.fromkeys(prediction.get(field, [])))
      truth = row[field]
      found += len(predicted)
      expected += len(truth)
      right_found += sum(any(same(p, t) for t in truth) for p in predicted)
      right_expected += sum(any(same(p, t) for p in predicted) for t in truth)
    precision = right_found / max(found, 1)
    recall = right_expected / max(expected, 1)
    rows[field] = {
      "precision": precision,
      "recall": recall,
      "F1": 2 * precision * recall / max(precision + recall, 1e-9),
    }
  return pd.DataFrame(rows).T.round(2)


results = {}

## 1. spaCy

Load `fr_core_news_md` (the passages are French) and extract each passage's entities: `PER` for people, `LOC` for places, `ORG` for organizations. Evaluate, and store the table in `results["spaCy"]`. How long for the 40 passages?

**Useful documentation**

- [`Language.pipe`](https://spacy.io/api/language#pipe)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `spacy.load("fr_core_news_md")`, then `nlp.pipe(gold.text)`: a `Doc` per passage, faster than one by one
- `doc.ents`: a passage's entities; `ent.text` is their text, `ent.label_` their label (`PER`, `LOC`, `ORG`)
- the format `evaluate` expects: a list of dictionaries `{"people": [...], "places": [...], "organizations": [...]}`, in the order of `gold`
- `time.time()` before and after to time it

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
import time

import spacy

nlp = spacy.load(...)
LABELS = {"PER": "people", "LOC": "places", "ORG": "organizations"}
start = time.time()
spacy_predictions = []
for doc in nlp.pipe(...):
  prediction = {field: [] for field in FIELDS}
  for ent in doc.ents:
    if ent.label_ in LABELS:
      prediction[LABELS[ent.label_]].append(...)
  spacy_predictions.append(prediction)
print(f"{time.time() - start:.1f} s")
results["spaCy"] = evaluate(...)
results["spaCy"]
```

### Solution

In [ ]:
import time

import spacy

nlp = spacy.load("fr_core_news_md")
LABELS = {"PER": "people", "LOC": "places", "ORG": "organizations"}
start = time.time()
spacy_predictions = []
for doc in nlp.pipe(gold.text):
  prediction = {field: [] for field in FIELDS}
  for ent in doc.ents:
    if ent.label_ in LABELS:
      prediction[LABELS[ent.label_]].append(ent.text)
  spacy_predictions.append(prediction)
print(f"{time.time() - start:.1f} s")
results["spaCy"] = evaluate(spacy_predictions)
results["spaCy"]

## 2. GLiNER

[GLiNER](https://github.com/urchade/GLiNER) finds the entities of the labels you give it, without training. Load `urchade/gliner_multi-v2.1` (on the GPU if there is one: `DEVICE`) and extract the entities of the labels `["person", "place", "organization"]` (`predict_entities`). Evaluate, and store the table in `results["GLiNER"]`, the extractions in `gliner_predictions` (exercise 4 uses them). Then try other labels ("newspaper", "political party"…, stored in `organizations`) or another threshold: which fields gain?

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `GLiNER.from_pretrained("urchade/gliner_multi-v2.1").to(DEVICE)`: loads the model, then puts it on the GPU (`DEVICE` is `"cpu"` without one)
- `gliner.predict_entities(text, labels, threshold=0.4)`: returns a list of dictionaries, with the entity's text (`"text"`) and its label (`"label"`); `threshold` is the minimum confidence (0.5 by default); 0.4, a common reference to compare the trials with each other, isn't necessarily the best threshold: find it by varying it
- a dictionary `FIELD_OF = {"person": "people", ...}` converts each GLiNER label into an `evaluate` field
- a function `extract_gliner(field_of, threshold)` that returns the extractions of the 40 passages: a second run is then a single call
- `{**FIELD_OF, "newspaper": "organizations"}`: the same dictionary, with one more label

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from gliner import GLiNER

gliner = GLiNER.from_pretrained(...).to(DEVICE)


def extract_gliner(
  field_of: dict[str, str], threshold: float
) -> list[dict[str, list[str]]]:
  predictions = []
  for text in gold.text:
    prediction = {field: [] for field in FIELDS}
    for entity in gliner.predict_entities(text, list(field_of), threshold=threshold):
      prediction[field_of[entity["label"]]].append(entity[...])
    predictions.append(prediction)
  return predictions


FIELD_OF = {"person": "people", "place": "places", "organization": "organizations"}
start = time.time()
gliner_predictions = extract_gliner(FIELD_OF, threshold=...)
print(f"{time.time() - start:.1f} s")
results["GLiNER"] = evaluate(...)
print(results["GLiNER"])

# A second run: other labels, or another threshold
more_labels = {**FIELD_OF, ...: "organizations"}
evaluate(extract_gliner(more_labels, threshold=...))
```

### Solution

In [ ]:
from gliner import GLiNER

gliner = GLiNER.from_pretrained("urchade/gliner_multi-v2.1").to(DEVICE)


def extract_gliner(
  field_of: dict[str, str], threshold: float
) -> list[dict[str, list[str]]]:
  predictions = []
  for text in gold.text:
    prediction = {field: [] for field in FIELDS}
    for entity in gliner.predict_entities(text, list(field_of), threshold=threshold):
      prediction[field_of[entity["label"]]].append(entity["text"])
    predictions.append(prediction)
  return predictions


FIELD_OF = {"person": "people", "place": "places", "organization": "organizations"}
start = time.time()
gliner_predictions = extract_gliner(FIELD_OF, threshold=0.4)
print(f"{time.time() - start:.1f} s")
results["GLiNER"] = evaluate(gliner_predictions)
print(results["GLiNER"])

# Changing labels takes no training: that is what spaCy, whose labels are fixed at
# training time, cannot do. With "newspaper" and "political party", people goes from
# 0.74 to 0.82 F1 and organizations from 0.60 to 0.65, while places drops from 0.85 to
# 0.83. A threshold of 0.6 with the three labels gives people 0.78, places 0.82 and
# organizations 0.65, but with the five, places falls to 0.76: settings are measured
# together, on your own data. Exercise 5's table keeps the first run.
more_labels = {
  **FIELD_OF,
  "newspaper": "organizations",
  "political party": "organizations",
}
evaluate(extract_gliner(more_labels, threshold=0.4))

## 3. An LLM Filling in a Schema

The model runs in Ollama, which offers an OpenAI-compatible API: the same code would work with a commercial API, changing the address, the key and the model name.

Describe the expected result with a Pydantic class `Entities`: three lists of strings, and a date that can be `None` (`str | None`), which the model leaves empty rather than invent it. Then write `extract(text)` that asks the model for the passage's entities, with `client.chat.completions.parse`: the answer follows the schema, unless it's cut off (`max_tokens`) or refused. Your prompt must give the annotation guidelines (below) and the date's format (ISO, `null` if the passage gives none). Run the extraction of the passages of `llm_texts` (the 40; without a GPU, the first 5) **in parallel**, time it, evaluate. The date isn't evaluated (`gold.jsonl` has none): read it on a few passages, against the issue's date, in their identifier (`gold.id`).

Annotation guidelines: each entity named once, as written in the text (OCR errors included); people without their title (M., Mme, sir); places (countries, towns, streets, buildings); named organizations (assemblies, parties, unions, newspapers, agencies); no generic mentions ("le gouvernement").

**Useful documentation**

- [`BaseModel.model_dump`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel.model_dump)
- [`ThreadPoolExecutor.map`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)

In [ ]:
pull.wait()  # the model is downloaded

from openai import OpenAI
from pydantic import BaseModel

# timeout: without a GPU, Ollama handles the requests one by one, and the last of the
# requests sent together waits for all the others (the default timeout is 600 s).
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=1800)
# A first call loads the model (onto the GPU if there is one), where it stays: before any parallel work.
client.chat.completions.create(
  model=MODEL, messages=[{"role": "user", "content": "Hello"}], max_tokens=1
)
print(MODEL, "ready")

# The passages the LLM processes: the 40 on a GPU; without one, 5 (about ten minutes).
llm_texts = gold.text if GPU else gold.text[:5]

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `class Entities(BaseModel)`: three fields, `people`, `places` and `organizations`, of type `list[str]`, and `date`, of type `str | None` (`None` if the passage gives none)
- `client.chat.completions.parse(model=MODEL, messages=[{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": text}], response_format=Entities, temperature=0, reasoning_effort="none")`
- `reasoning_effort="none"`: the model answers without a thinking phase (without a GPU, Qwen3-1.7B would have one, which multiplies the response time)
- `response.choices[0].message.parsed.model_dump()`: the parsed object, converted into a dictionary
- `ThreadPoolExecutor(8)`: 8 calls in parallel; `list(pool.map(extract, llm_texts))` returns the extractions in the passages' order
- `zip(llm_predictions[:5], gold.id)`: the first 5 extractions, with their passage's identifier, which holds the issue's date

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from concurrent.futures import ThreadPoolExecutor


class Entities(BaseModel):
  people: list[str]
  places: ...
  organizations: ...
  date: ...


INSTRUCTIONS = """Extract the named entities of this passage of the 1914 French press.
- people: persons, without their title (M., Mme, sir, lord);
- places: places (countries, regions, towns, streets, buildings);
- organizations: named organizations (assemblies, parties, unions, newspapers, agencies).
- date: the date of the events, in ISO format (YYYY-MM-DD), null if the passage gives none.
Each entity once, as written in the text, errors included.
No generic mentions such as "le gouvernement". Do not invent anything."""


def extract(text: str) -> dict[str, list[str] | str | None]:
  response = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "system", "content": ...}, {"role": "user", "content": ...}],
    response_format=...,
    temperature=0,
    reasoning_effort="none",
  )
  return response.choices[0].message.parsed.model_dump()


start = time.time()
with ThreadPoolExecutor(8) as pool:
  llm_predictions = list(pool.map(..., llm_texts))
print(f"{time.time() - start:.1f} s")
# The date, against the issue's (in the passage's identifier)
for prediction, passage_id in zip(llm_predictions[:5], gold.id):
  print(f"{passage_id:26} {prediction[...]}")
results["LLM"] = evaluate(...)
results["LLM"]
```

### Solution

In [ ]:
from concurrent.futures import ThreadPoolExecutor


class Entities(BaseModel):
  people: list[str]
  places: list[str]
  organizations: list[str]
  date: str | None


INSTRUCTIONS = """Extract the named entities of this passage of the 1914 French press.
- people: persons, without their title (M., Mme, sir, lord);
- places: places (countries, regions, towns, streets, buildings);
- organizations: named organizations (assemblies, parties, unions, newspapers, agencies).
- date: the date of the events, in ISO format (YYYY-MM-DD), null if the passage gives none.
Each entity once, as written in the text, errors included.
No generic mentions such as "le gouvernement". Do not invent anything."""


def extract(text: str) -> dict[str, list[str] | str | None]:
  response = client.chat.completions.parse(
    model=MODEL,
    messages=[
      {"role": "system", "content": INSTRUCTIONS},
      {"role": "user", "content": text},
    ],
    response_format=Entities,
    temperature=0,
    reasoning_effort="none",
  )
  return response.choices[0].message.parsed.model_dump()


start = time.time()
with ThreadPoolExecutor(8) as pool:
  llm_predictions = list(pool.map(extract, llm_texts))
print(f"{time.time() - start:.1f} s")
# The date isn't evaluated (gold.jsonl has none): we read it, against the issue's
# date, in the passage's identifier.
for prediction, passage_id in zip(llm_predictions[:5], gold.id):
  print(f"{passage_id:26} {prediction['date']}")
results["LLM"] = evaluate(llm_predictions)
results["LLM"]

## 4. Link Names to a Reference List

The same people appear in different forms ("Jaurès", "Jean Jaurès", "M. Jaurès"), sometimes misread by the OCR. Link the people GLiNER found (`gliner_predictions`, the 40 passages, with or without a GPU) to `REFERENCE`: for each one, the closest name (`rapidfuzz.process.extractOne`), kept if its score reaches a threshold of 85, then of 75. What do you gain, what do you lose? Find a false link, and a rule that rules it out.

**Useful documentation**

- [`rapidfuzz.process.extractOne`](https://rapidfuzz.github.io/RapidFuzz/Usage/process.html#extractone)
- [`rapidfuzz.fuzz.WRatio`](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html#wratio)
- [`rapidfuzz.fuzz.ratio`](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html#ratio)

In [ ]:
REFERENCE = [
  "Jean Jaurès",
  "Joseph Caillaux",
  "Henriette Caillaux",
  "Gaston Calmette",
  "Raymond Poincaré",
  "René Viviani",
  "Georges Clemenceau",
  "Jules Guesde",
  "Léon Jouhaux",
  "Édouard Vaillant",
  "Edward Grey",
  "Herbert Asquith",
  "Paul Cambon",
  "Raoul Villain",
  "Gabriel Hanotaux",
  "Antonino di San Giuliano",
  "Émile Vandervelde",
  "John Simon",
  "Woodrow Wilson",
]

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `process.extractOne(name, REFERENCE, scorer=fuzz.WRatio)` returns a tuple `(choice, score, position)`; `WRatio` combines several measures and tolerates word order and partial names
- `sorted({name for prediction in gliner_predictions for name in prediction["people"]})`: the people GLiNER extracted, without duplicates
- for the rule: the extracted name's length, or its last word (`name.split()[-1]`) compared with the reference name's with `fuzz.ratio`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from rapidfuzz import process

people = sorted({name for prediction in ... for name in prediction["people"]})
for threshold in (85, 75):
  print(f"--- threshold of {threshold}")
  for name in people:
    match, score, _ = process.extractOne(name, ..., scorer=...)
    if score >= threshold:
      print(f"{name:25} -> {match} ({score:.0f})")


def link(name: str, threshold: float = 75) -> str | None:
  match, score, _ = process.extractOne(name, REFERENCE, scorer=fuzz.WRatio)
  rule = ...  # a condition that rules out the false link
  return match if score >= threshold and rule else None


print("--- threshold of 75, with the rule")
for name in people:
  if link(name) is not None:
    print(f"{name:25} -> {link(name)}")
```

### Solution

In [ ]:
from rapidfuzz import process

people = sorted(
  {name for prediction in gliner_predictions for name in prediction["people"]}
)
for threshold in (85, 75):
  print(f"--- threshold of {threshold}")
  for name in people:
    match, score, _ = process.extractOne(name, REFERENCE, scorer=fuzz.WRatio)
    if score >= threshold:
      print(f"{name:25} -> {match} ({score:.0f})")

# At 85, 12 links, one of them false: il -> Joseph Caillaux (90), since WRatio gives 90
# to a fragment ("il" is in "Caillaux"). At 75, 3 more: M. Calmette and M. Cambon,
# right, and Reymond -> Raymond Poincaré, wrong (Doctor Reymond). Lowering the threshold
# catches variants, and merely similar names with them. A rule: the last word (the
# surname) must itself resemble the reference's. It rules out both false links (a
# minimum length would only rule out "il"), and keeps "£1 Asquith", where the OCR read
# "£1" for "M.".


def link(name: str, threshold: float = 75) -> str | None:
  match, score, _ = process.extractOne(name, REFERENCE, scorer=fuzz.WRatio)
  same_surname = (
    fuzz.ratio(normalize(name.split()[-1]), normalize(match.split()[-1])) >= 85
  )
  return match if score >= threshold and same_surname else None


print("--- threshold of 75, with the rule")
for name in people:
  if link(name) is not None:
    print(f"{name:25} -> {link(name)}")

## 5. Compare, and Spot Inventions

Gather the three methods' F1 in one table. Then look for the LLM's **inventions**: extracted entities that don't appear in the passage's text (measured with `fuzz.partial_ratio`). Are there any? Read each one again in its passage. Which of the three methods would you choose for 10 million passages?

Without a GPU, the next cell replaces the extractions of the 5 passages with those of the 40, made by Qwen3-4B on a GPU with exercise 3's guidelines (without the date): the table and the search then cover the 40 passages, as with a GPU.

**Useful documentation**

- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`rapidfuzz.fuzz.partial_ratio`](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html#partial-ratio)

In [ ]:
# Without a GPU, exercise 3 only extracted 5 passages: we load the extractions of the
# 40, made on a GPU by Qwen3-4B with the same guidelines (without the date).
if not GPU:
  teacher = pd.read_json("teacher-gold.jsonl", lines=True).set_index("id")
  llm_predictions = teacher.loc[gold.id, FIELDS].to_dict("records")
  results["LLM"] = evaluate(llm_predictions)
print(len(llm_predictions), "passages extracted by the LLM")

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `pd.DataFrame({name: table["F1"] for name, table in results.items()})`: one column per method
- `fuzz.partial_ratio(normalize(entity), normalize(text))`: the best similarity between the short string and a part of the long one; under 85, the entity isn't in the text

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
print(pd.DataFrame({name: ... for name, table in results.items()}))

for prediction, text in zip(llm_predictions, gold.text):
  for field in FIELDS:
    for entity in prediction[field]:
      if fuzz.partial_ratio(normalize(...), normalize(...)) < ...:
        print(f"{field:13} {entity!r} is not in the text")
```

### Solution

In [ ]:
print(pd.DataFrame({name: table["F1"] for name, table in results.items()}))

for prediction, text in zip(llm_predictions, gold.text):
  for field in FIELDS:
    for entity in prediction[field]:
      if fuzz.partial_ratio(normalize(entity), normalize(text)) < 85:
        print(f"{field:13} {entity!r} is not in the text")

# With the Qwen3-4B extractions loaded without a GPU, 4 entities out of 257 are
# flagged, and only one is really missing from the text ("gouvernement"). "école du
# Louvre" is there, cut by the OCR ("Lou- •;r e"); "le chancelier" scores 84.6 ("au
# chancelier"); "l'homme" is there ("un homme"), even though it isn't a name. On
# noisy text, a string measure flags false inventions: every flag gets read again in
# its passage.

# spaCy is by far the fastest, but its labels are fixed and it errs on noisy text. Here
# GLiNER does at least as well as the LLM on people and places (the gap stays within
# the noise of 40 passages), at a far lower cost. The LLM's strength is its
# flexibility (a schema, free-form fields, not only entities), not its accuracy: measure on
# your own data. A larger model might change this result. For 10 million passages, we'd
# keep GLiNER; for a schema it can't fill, we'd use the LLM to label examples, then
# fine-tune a small model (fine-tuning lab, later on).